# Week 4 Lab — Transformer Blocks from Scratch (PyTorch)

**Goals**
- Implement scaled dot-product self-attention.
- Implement causal masking (decoder-style).
- Build a minimal Transformer block: (LN → Attn → Residual → LN → MLP → Residual).
- Inspect attention matrices and verify masking.

This lab is intentionally small and inspectable; it’s about understanding *mechanics*, not achieving SOTA performance.


## 0. Setup
CPU is fine. If you have a GPU, it will be used automatically.


In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. Toy inputs
We’ll use random token embeddings so we can focus on shape correctness and masking.

Notation:
- `B`: batch size
- `T`: sequence length
- `D`: model dimension


In [ ]:
B, T, D = 2, 8, 16
X = torch.randn(B, T, D, device=device)
X.shape

## 2. Scaled dot-product attention (single head)

Attention weights:
\[
A = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
\]

Output:
\[
Y = AV
\]


In [ ]:
def scaled_dot_product_attention(Q, K, V, attn_mask=None):
    """Single-head attention.
    Q, K, V: [B, T, d_k]
    attn_mask: [T, T] or [B, T, T] with True where attention is allowed.
    """
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)  # [B, T, T]

    if attn_mask is not None:
        # attn_mask True = allow, False = disallow
        if attn_mask.dim() == 2:
            mask = attn_mask.unsqueeze(0)  # [1, T, T]
        else:
            mask = attn_mask
        scores = scores.masked_fill(~mask, float('-inf'))

    A = torch.softmax(scores, dim=-1)  # [B, T, T]
    Y = A @ V  # [B, T, d_k]
    return Y, A

# quick smoke test
d_k = D
Wq = torch.randn(D, d_k, device=device)
Wk = torch.randn(D, d_k, device=device)
Wv = torch.randn(D, d_k, device=device)
Q, K, Vv = X @ Wq, X @ Wk, X @ Wv
Y, A = scaled_dot_product_attention(Q, K, Vv)
Y.shape, A.shape

## 3. Causal (decoder) mask

A decoder-only LLM must not attend to future tokens.
Create a lower-triangular mask allowing attention only to positions `<= t`.


In [ ]:
def causal_mask(T: int, device=None):
    m = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))
    return m

mask = causal_mask(T, device=device)
mask

In [ ]:
Yc, Ac = scaled_dot_product_attention(Q, K, Vv, attn_mask=mask)
Yc.shape, Ac.shape

### Verify masking
Check that probabilities above the diagonal are ~0.


In [ ]:
# For each query position t, Ac[..., t, j] should be ~0 for j > t
upper = torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)
leak = Ac.masked_select(upper).abs().max().item()
leak

## 4. Multi-head attention (minimal)

We implement multi-head attention by splitting the model dimension into `H` heads.

Shapes:
- input `X`: [B, T, D]
- per-head dimension `d_h = D / H`
- attention performed per head


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x):
        # x: [B, T, D] -> [B, H, T, d_head]
        B, T, D = x.shape
        x = x.view(B, T, self.n_heads, self.d_head)
        return x.permute(0, 2, 1, 3)

    def _merge_heads(self, x):
        # x: [B, H, T, d_head] -> [B, T, D]
        B, H, T, d_head = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, T, H * d_head)

    def forward(self, x, attn_mask=None, return_attn=False):
        # x: [B, T, D]
        Q = self._split_heads(self.Wq(x))
        K = self._split_heads(self.Wk(x))
        V = self._split_heads(self.Wv(x))

        # reshape to combine batch and heads for attention
        B, H, T, d_h = Q.shape
        Q2 = Q.reshape(B * H, T, d_h)
        K2 = K.reshape(B * H, T, d_h)
        V2 = V.reshape(B * H, T, d_h)

        if attn_mask is not None and attn_mask.dim() == 2:
            m = attn_mask.unsqueeze(0).expand(B * H, -1, -1)
        else:
            m = attn_mask

        Y2, A2 = scaled_dot_product_attention(Q2, K2, V2, attn_mask=m)
        Y = Y2.view(B, H, T, d_h)
        Y = self._merge_heads(Y)
        out = self.Wo(Y)
        if return_attn:
            A = A2.view(B, H, T, T)
            return out, A
        return out

mha = MultiHeadSelfAttention(d_model=D, n_heads=4).to(device)
out, attn = mha(X, attn_mask=mask, return_attn=True)
out.shape, attn.shape

## 5. Transformer block (decoder-style, minimal)

Block structure (pre-LN style):
- LayerNorm
- Masked self-attention
- Residual
- LayerNorm
- MLP
- Residual


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, d_ff)

    def forward(self, x, attn_mask=None, return_attn=False):
        if return_attn:
            a_in = self.ln1(x)
            attn_out, A = self.attn(a_in, attn_mask=attn_mask, return_attn=True)
            x = x + attn_out
            x = x + self.mlp(self.ln2(x))
            return x, A
        x = x + self.attn(self.ln1(x), attn_mask=attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x

block = TransformerBlock(d_model=D, n_heads=4, d_ff=64).to(device)
Y, A = block(X, attn_mask=mask, return_attn=True)
Y.shape, A.shape

## 6. (Optional) Tiny next-token model (1–2 blocks)

If you want to go one step further, you can build a minimal decoder-only model:
- token embedding
- positional embedding
- 1–2 transformer blocks
- linear head to vocab

Then train on a toy dataset (like Week 1) for a few minutes.

This is optional — Week 4 is primarily about architecture correctness.


## 7. Reflection (required)

Answer briefly:
- What does attention compute, operationally?
- Why must masking be correct in decoder models?
- Where does positional information enter (conceptually)?
- Why are residual connections essential for deep stacks?

Add any notes about surprising results from attention matrix inspection.
